In [ ]:
import pandas as pd
import re
df = pd.read_csv(r'$DATA',sep=';')
def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.strip()
    if (text.startswith('"') and text.endswith('"')) or (text.startswith("'") and text.endswith("'")):
        text = text[1:-1]
        text = text.strip()
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\sáéíóúÁÉÍÓÚüÜñÑ,:;()/-]', '', text)
    return text.strip()

df['TEXTO_LIMPIO'] = df['TEXTO'].apply(clean_text)

initial_count = len(df)
df = df[df['TEXTO_LIMPIO'].str.strip() != ""]
df = df[df['TEXTO_LIMPIO'].str.len() >= 10]
df = df.drop_duplicates(subset=['ID_PACIENTE', 'TEXTO_LIMPIO'])
print(f"Original number of documents: {initial_count}")
print(f"Documents after cleaning: {len(df)}")
print(df['RECURRENCIA'].value_counts())
df_clean = df[['ID_PACIENTE', 'TEXTO_LIMPIO', 'RECURRENCIA', 'FOLD']]
df_clean.to_csv('../data/recurrence_data_clean.csv', index=False)
print("Cleaned dataset saved as recurrence_data_clean.csv")




In [ ]:
import pandas as pd
df_original = pd.read_csv('../data/.csv', sep=';')
df_clean = pd.read_csv('../data/.csv', dtype=str)
df_original.columns = [col.strip().upper() for col in df_original.columns]
df_clean.columns = [col.strip().upper() for col in df_clean.columns]


def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.strip()
    if (text.startswith('"') and text.endswith('"')) or (text.startswith("'") and text.endswith("'")):
        text = text[1:-1]
        text = text.strip()
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\sáéíóúÁÉÍÓÚüÜñÑ,:;()/-]', '', text)
    return text.strip()

df_original['TEXTO_LIMPIO'] = df_original['TEXTO'].apply(clean_text)

cols = ['ID_PACIENTE', 'TEXTO_LIMPIO', 'RECURRENCIA', 'FOLD']
for col in ['ID_PACIENTE', 'TEXTO_LIMPIO', 'RECURRENCIA', 'FOLD']:
    df_original[col] = df_original[col].astype(str)
    df_clean[col] = df_clean[col].astype(str)

df_removed = pd.merge(
    df_original[cols],
    df_clean[cols],
    on=cols,
    how='left',
    indicator=True
).query('_merge == "left_only"').drop('_merge', axis=1)

print(f"Rows removed during cleaning: {len(df_removed)}")

df_removed.to_csv('../data/rows_removed_in_cleaning.csv', index=False)
print("Removed rows saved to rows_removed_in_cleaning.csv")


In [ ]:
import pandas as pd
import os

INPUT_CSV = r".csv"
OUTPUT_CSV = r".csv"


df = pd.read_csv(INPUT_CSV, encoding='utf-8')
def split_patient_id(id_string):
    if pd.isna(id_string) or not isinstance(id_string, str):
        return pd.Series({
            'id_paciente': None,
            'id_consulta': None,
            'fecha': None
        })
    
    id_paciente = id_string[:36]
    fecha = id_string[-10:]
    id_consulta_raw = id_string[36:-10]
    id_consulta = id_consulta_raw.strip('_')
    
    return pd.Series({
        'id_paciente': id_paciente,
        'id_consulta': id_consulta,
        'fecha': fecha
    })

df[['id_paciente', 'id_consulta', 'fecha']] = df['ID_PACIENTE'].apply(split_patient_id)
cols = ['id_paciente', 'id_consulta', 'fecha'] + [col for col in df.columns if col not in ['id_paciente', 'id_consulta', 'fecha', 'ID_PACIENTE']]
df_reordered = df[cols]
for idx in range(min(3, len(df_reordered))):
    print(f"\nFila {idx + 1}:")
    print(f"  id_paciente:  {df_reordered.iloc[idx]['id_paciente']}")
    print(f"  id_consulta:  {df_reordered.iloc[idx]['id_consulta']}")
    print(f"  fecha:        {df_reordered.iloc[idx]['fecha']}")
if df_reordered['id_paciente'].notna().any():
    print(f"\n  Longitud id_paciente: {df_reordered['id_paciente'].str.len().mode().values[0]}")
if df_reordered['fecha'].notna().any():
    print(f"  Longitud fecha: {df_reordered['fecha'].str.len().mode().values[0]}")
df_reordered.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')



In [ ]:
import pandas as pd
import os
import re


INPUT_CSV = ".csv"
OUTPUT_CSV = ".csv"

print("Cargando datos...")
df = pd.read_csv(INPUT_CSV, encoding='utf-8')

print(f"Total documentos: {len(df)}")
print(f"Pacientes unicos: {df['id_paciente'].nunique()}")

# Convertir fecha a datetime para ordenar
df['fecha_dt'] = pd.to_datetime(df['fecha'], format='%Y/%m/%d', errors='coerce')
df = df.sort_values(['id_paciente', 'fecha_dt']).reset_index(drop=True)

def clean_sequential(group):
    cleaned_texts = []
    
    for idx, (orig_idx, row) in enumerate(group.iterrows()):
        current_text = row['TEXTO_LIMPIO']
        
        if pd.isna(current_text) or not isinstance(current_text, str):
            cleaned_texts.append(current_text)
            continue
        
        cleaned_text = current_text
        
        if idx > 0 and len(cleaned_texts) > 0:
            previous_text = cleaned_texts[idx - 1]  # Usar texto ya limpiado de la anterior
            
            if pd.notna(previous_text) and isinstance(previous_text, str) and previous_text.strip():
                if previous_text.strip() in cleaned_text:
                    cleaned_text = cleaned_text.replace(previous_text.strip(), '')
                    
                    # Limpiar espacios y saltos de linea sobrantes
                    cleaned_text = re.sub(r'\n\s*\n+', '\n', cleaned_text)
                    cleaned_text = re.sub(r' +', ' ', cleaned_text)
                    cleaned_text = cleaned_text.strip()
        
        cleaned_texts.append(cleaned_text)
        
        if cleaned_text != current_text:
            original_len = len(current_text)
            cleaned_len = len(cleaned_text)
            
            print(f"Paciente: {row['id_paciente']}")
            print(f"  Consulta: {row['fecha']} ({row['id_consulta'][:8]}...)")
            print(f"  Consulta numero: {idx + 1}")
            print(f"  Caracteres originales: {original_len}")
            print(f"  Caracteres limpiados: {cleaned_len}")
            print(f"  Caracteres eliminados: {original_len - cleaned_len}")
            print()
    
    return cleaned_texts

modified_count = 0
for patient_id, group in df.groupby('id_paciente'):
    if len(group) > 1:
        cleaned_texts = clean_sequential(group)
        for orig_idx, clean_text in zip(group.index, cleaned_texts):
            orig_text = df.loc[orig_idx, 'TEXTO_LIMPIO']
            df.loc[orig_idx, 'TEXTO_LIMPIO'] = clean_text
            
            if orig_text != clean_text:
                modified_count += 1

df = df.drop(columns=['fecha_dt'])
print(f"\nGuardando resultado en: {OUTPUT_CSV}")
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f"\nResumen:")
print(f"  Total consultas modificadas: {modified_count}")
print(f"  Total consultas: {len(df)}")
print(f"  Porcentaje modificado: {modified_count/len(df)*100:.1f}%")
print(f"\nArchivo guardado: {os.path.abspath(OUTPUT_CSV)}")
